# 01 — Flight Price Regression
## Project summary
**Business goal:** estimate flight price from historical flight attributes.  
**GitHub repository:** `INSERT_SINGLE_GITHUB_REPOSITORY_URL_HERE`  
**Dataset:** `../data/flights.csv`  
This notebook is an exploratory and explanatory companion to the reusable training script. The deployed model is trained through `src/travel_ml/train_regression.py` so the same pipeline can be used by the API.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
df = pd.read_csv(ROOT / "data" / "flights.csv")
df.shape, df.head(), df.isna().sum()

## Data understanding
The flight table contains route, flight type, agency, time, distance, date, identifiers, and target price. Identifiers are not used as predictors because they can let a model memorize rows instead of learning price patterns. The schema does not specify currency or the units for time and distance, so conclusions should remain in dataset units.

In [ ]:
df.describe(include="all").T

In [ ]:
df["price"].hist(bins=50)
plt.title("Flight price distribution")
plt.xlabel("Price (dataset units)")
plt.ylabel("Rows")
plt.show()

In [ ]:
for col in ["flightType", "agency", "from", "to"]:
    print("\\n", col, "unique:", df[col].nunique())
    display(df[col].value_counts().head(10).to_frame("count"))

## Feature selection and validation
Features: origin, destination, flight type, agency, time, distance, and date-derived calendar values. The training pipeline imputes numeric missing values, one-hot encodes categories with unknown-category handling, and scales numeric features. The provided data has no missing values in the initial audit, but reusable preprocessing makes inference more robust.

The script uses a reproducible random holdout as a baseline. If the business goal is to estimate future fares, prefer chronological validation and check whether booking date and travel date are available at prediction time.

In [ ]:
from src.travel_ml.train_regression import train
model, metadata = train(ROOT / "data" / "flights.csv", ROOT / "artifacts", sample_size=50000)
metadata

## Interpretation
Read the actual MAE, RMSE, and R² printed by the run. MAE is the average absolute error in price units; RMSE penalizes large misses more heavily; R² measures variance explained relative to a mean baseline. Do not report invented metrics. Compare with a `DummyRegressor` baseline and inspect residuals before claiming the model is useful.

In [ ]:
import json
metrics_path = ROOT / "artifacts" / "flight_price_metrics.json"
json.loads(metrics_path.read_text())

## Conclusion and limitations
The pipeline is deployable because preprocessing and the estimator are serialized together. Next steps are chronological backtesting, tuning, prediction interval estimation, drift monitoring, and a model-promotion gate. The model is not a live airline pricing engine and cannot guarantee an actual fare.